In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torch

import neurogym as ngym
from neurogym.wrappers import ScheduleEnvs, Monitor
from neurogym.utils import plot_env, RandomSchedule, SequentialSchedule
from neurogym.envs import get_collection
from torch.nn import functional as F

from IPython.display import clear_output
from net.common import savefig
warnings.filterwarnings("ignore")
clear_output()

In [ ]:
# Environment parameters
# This setting is low to speed up testing, we recommend setting it to at least 1000
EVAL_TRIALS = 100
dt = 100
dim_ring = 2  # Number of choices in the ring representation
kwargs = {
    "dt": dt,
    "dim_ring": dim_ring,
}
rewards = {
    "abort": -0.1,
    "correct": +1.0,
    "fail": 0.0
}
sigma = 1.0 # Standard deviation of the Gaussian noise in the ring representation
device = 'cuda' if torch.cuda.is_available() else 'cpu'
ob_size = 5
act_size = 3
print(f"ob_size: {ob_size}, act_size: {act_size}"   )

In [ ]:
task_ids = ["go", "anti", "dlygo", "dlyanti"]
task = [f"yang19.{task}-v0" for task in task_ids]
envs = [ngym.make(t, **kwargs) for t in task]
timing_idx = {'stimulus':[], 'decision':[]}
for i in range(len(envs)):
    timing = envs[i].timing
    timing_idx["stimulus"].append((int(timing['fixation']/dt), int((timing['fixation']+timing['stimulus'])/dt)))
    decision_start = timing['fixation'] + timing['stimulus'] + timing['delay']
    decision_end = timing['fixation'] + timing['stimulus'] + timing['delay'] + timing['decision']
    timing_idx['decision'].append((int(decision_start/dt), int(decision_end/dt)))
    print(timing)
print(timing_idx)


In [ ]:
import pickle as pkl
model_names = ['eisep', 'simple']
legend_names = ['EISep', 'Simple']
lrs = np.array([0.5, 1e-1, 5e-2, 1e-2, 5e-3, 1e-3, 5e-4, 1e-4])
n_seed = 5
epochs = 3000
hidden_size = 64
len_multilr_envs = 20
n_lr = len(lrs)
test_acc = np.zeros((len(model_names), n_lr, n_seed, len_multilr_envs))
net_state_dicts = []
# colors = ['C1', 'C0', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9']
# colors = ['C1', 'C0', 'C7', 'C8', 'C9', '#E69F00', '#56B4E9']
colors = ['C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', '#E69F00', '#56B4E9']

for i, model_name in enumerate(model_names):
    print(i)
    path = f'./result/multitask/multilr_{model_name}_performance.pkl'
    with open(path, 'rb') as f:
        info = pkl.load(f)
    test_acc[i] = info['performance']
# print(env_names)
test_acc_flatten = test_acc.reshape(len(model_names), len(lrs), -1)
mean_acc = np.mean(test_acc_flatten, axis=-1)
best_lr_idx = np.argmax(mean_acc, axis=-1)
best_lrs = lrs[best_lr_idx]
print(f"Best lrs: {best_lrs}")
best_lr_dict = {model_name: float(best_lr) for model_name, best_lr in zip(model_names, best_lrs)}
print(best_lr_dict)

lrs = [best_lr_dict[model_name] for model_name in model_names]
# paths = [f'./result/continual_learning_reweight/hidden_64/lr{lr}_{model_name}_info.pkl' for model_name, lr in zip(model_names, lrs)]
paths = [f'./result/continual_learning_gradient_alignment_reorder/lr{lr}_{model_name}_info.pkl' for model_name, lr in zip(model_names, lrs)]

for i in range(len(paths)):
    print(paths[i])
infos = [pkl.load(open(path, 'rb')) for path in paths]
test_perfs = [info['test_perf_all'] for info in infos]
print('EISep:')
for k,v in infos[0].items():
    print(k, np.array(v).shape)
print('Simple:')
for k,v in infos[1].items():
    print(k, np.array(v).shape)

In [ ]:
def collect_hidden_states(env, net, env_vec, num_trial=100, stimulus_idx=None, decision_idx=None):
    """
    收集网络在某个任务上的隐层激活。
    返回 shape: (total_timesteps, hidden_size)
    """
    net.eval()
    all_hidden = []
    for i in range(num_trial):
        env.seed(i)
        env.new_trial()
        ob, gt = env.ob, env.gt
        ob = ob[:, np.newaxis, :]  # (T, 1, ob_size)
        ob = np.concatenate(
            [ob, env_vec.reshape(1, 1, -1).repeat(ob.shape[0], 0).repeat(ob.shape[1], 1)],
            axis=-1
        )
        inputs = torch.from_numpy(ob).type(torch.float).to(device)
        with torch.no_grad():
            _, _, hidden = net(inputs)  # hidden: [(T, 1, hidden_size),...]
        all_hidden.append(hidden[0].squeeze(1).cpu().numpy())  # (T, hidden_size)
    stacked_all_hidden = np.stack(all_hidden, 0) # (num_trial, T, hidden_size)
    stimulus_hidden = stacked_all_hidden[:, stimulus_idx[0]:stimulus_idx[1]]
    decision_hidden = stacked_all_hidden[:, decision_idx[0]:decision_idx[1]]
    return stimulus_hidden, decision_hidden


In [ ]:
from net.common import init_model
num_trial = 100
stimulus_T = 5
decision_T = 5
stimulus_hidden_state_all = np.zeros((len(paths), n_seed, len(envs), num_trial, stimulus_T, hidden_size))
decision_hidden_state_all = np.zeros((len(paths), n_seed, len(envs), num_trial, decision_T, hidden_size))
env_vecs = np.eye(len(envs))
for i in range(len(paths)):
    print(model_names[i])
    for j in range(n_seed):
        print('Seed:', j)
        np.random.seed(j)
        torch.manual_seed(j)
        net = init_model(
            ob_size=ob_size + len(envs),
            hidden_size=hidden_size,
            act_size=act_size,
            model_name=model_names[i]
        ).to(device)
        # print(len(infos), len(infos[i]['net_state_dicts']), len(infos[i]['net_state_dicts'][j]), len(infos[i]['net_state_dicts'][j][-1]))

        for k, env_test in enumerate(envs):
            net.load_state_dict(infos[i]['net_state_dicts'][j][-1][k]) # j-th seed, last round, Delay Anti task (last)
            stimulus_hidden, decision_hidden = collect_hidden_states(env_test, net, env_vecs[k],
                            num_trial, timing_idx['stimulus'][k], timing_idx['decision'][k])
            stimulus_hidden_state_all[i, j, k] = stimulus_hidden
            decision_hidden_state_all[i, j, k] = decision_hidden

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# =========================================================
# Settings
# =========================================================
env_titles = ["Go", "Anti", "Delay Go", "Delay Anti"]
type_names = ["stimulus", "decision"]
model_titles = ["EISep", "Simple"]

# stimulus_hidden_state_all:
# (n_model, n_seed, n_env, n_trial, stimulus_T, hidden_size)
# decision_hidden_state_all:
# (n_model, n_seed, n_env, n_trial, decision_T, hidden_size)

hidden_dict = {
    "stimulus": stimulus_hidden_state_all,
    "decision": decision_hidden_state_all,
}

n_models = len(model_names)
n_envs = len(envs)
n_types = len(type_names)

# rows = model x type, cols = env
n_rows = n_models * n_types
n_cols = n_envs

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(4.2 * n_cols, 3.6 * n_rows),
    constrained_layout=True
)

# Make sure axes is always 2D
if n_rows == 1:
    axes = axes[None, :]
if n_cols == 1:
    axes = axes[:, None]

# =========================================================
# PCA + plot
# =========================================================
for model_idx in range(n_models):
    for type_idx, type_name in enumerate(type_names):
        row = model_idx * n_types + type_idx
        hidden_all = hidden_dict[type_name]
        # shape: (n_model, n_seed, n_env, n_trial, T_type, hidden_size)

        for env_idx in range(n_envs):
            ax = axes[row, env_idx]

            X = hidden_all[model_idx, :, env_idx]
            # X shape: (n_seed, n_trial, T_type, hidden_size)

            n_seed_, n_trial_, T_, H_ = X.shape

            # flatten to (N_points, hidden_size)
            X_flat = X.reshape(-1, H_)

            # time labels for coloring
            # repeated over seed and trial
            time_labels = np.tile(np.arange(T_), n_seed_ * n_trial_)

            # PCA
            pca = PCA(n_components=2)
            X_pca = pca.fit_transform(X_flat)

            sc = ax.scatter(
                X_pca[:, 0], X_pca[:, 1],
                c=time_labels,
                cmap="viridis",
                s=12,
                alpha=0.6,
                linewidths=0
            )

            # Titles for the top row only
            if row == 0:
                ax.set_title(env_titles[env_idx], fontsize=13)

            # Row labels on first column only
            if env_idx == 0:
                ax.set_ylabel(
                    f"{model_titles[model_idx]}\n{type_name}",
                    fontsize=12
                )

            evr = pca.explained_variance_ratio_
            ax.set_xlabel(f"PC1 ({evr[0]*100:.1f}%)", fontsize=10)
            ax.set_ylabel(ax.get_ylabel() + f"\nPC2 ({evr[1]*100:.1f}%)" if env_idx == 0 else f"PC2 ({evr[1]*100:.1f}%)", fontsize=10)

            ax.tick_params(axis='both', labelsize=9)

# =========================================================
# One shared colorbar
# =========================================================
cbar = fig.colorbar(sc, ax=axes, shrink=0.98, pad=0.01)
cbar.set_label("Time step within epoch", fontsize=12)

plt.suptitle("PCA of hidden states across environments, epochs, and models", fontsize=16, y=1.02)
plt.show()

In [ ]:
def one_step_hidden_update(net, h, x_const):
    """
    h:      (batch, hidden_size)
    x_const:(batch, input_size)
    returns h_next: (batch, hidden_size)
    """
    W_actual = net.get_actual_weight()

    input_term = net.lstm.input_layer(x_const)         # (B, H)
    rec_term   = h @ W_actual.T + net.lstm.bias        # (B, H)
    h_next     = net.lstm.act(input_term + rec_term)   # (B, H)
    return h_next


def find_fixed_points_from_states(net, x_const, h_init_states, n_iters=2000, lr=1e-2, tol=1e-8):
    net.eval()
    device = next(net.parameters()).device

    if x_const.ndim == 1:
        x_const = x_const.unsqueeze(0)
    x_const = x_const.to(device)

    h = torch.tensor(h_init_states, dtype=torch.float32, device=device)
    h = torch.nn.Parameter(h.clone())

    x_batch = x_const.repeat(h.shape[0], 1)
    optimizer = torch.optim.Adam([h], lr=lr)

    for _ in range(n_iters):
        optimizer.zero_grad()
        h_next = one_step_hidden_update(net, h, x_batch)
        q = 0.5 * ((h_next - h) ** 2).sum(dim=1)
        loss = q.mean()
        loss.backward()
        optimizer.step()

        if q.mean().item() < tol:
            break

    with torch.no_grad():
        h_final = h.detach().clone()
        h_next = one_step_hidden_update(net, h_final, x_batch)
        q_final = 0.5 * ((h_next - h_final) ** 2).sum(dim=1)

    return deduplicate_fixed_points(h_final, q_final)


def deduplicate_fixed_points(h_all, q_all, q_thresh=1e-6, dist_thresh=1e-3):
    """
    Keep only good fixed points (small q), then merge near-duplicates.
    """
    good = q_all < q_thresh
    h_good = h_all[good]
    q_good = q_all[good]

    if h_good.shape[0] == 0:
        return h_good, q_good, h_all, q_all

    kept_h = []
    kept_q = []

    for i in range(h_good.shape[0]):
        h_i = h_good[i]
        is_new = True
        for h_k in kept_h:
            if torch.norm(h_i - h_k).item() < dist_thresh:
                is_new = False
                break
        if is_new:
            kept_h.append(h_i)
            kept_q.append(q_good[i])

    fps = torch.stack(kept_h, dim=0)
    q_vals = torch.stack(kept_q, dim=0)
    return fps, q_vals, h_all, q_all


def fixed_point_jacobian(net, h_star, x_const):
    """
    h_star: (hidden_size,)
    x_const: (input_size,) or (1, input_size)
    returns J: (hidden_size, hidden_size)
    """
    net.eval()
    device = next(net.parameters()).device

    if x_const.ndim == 1:
        x_const = x_const.unsqueeze(0)
    x_const = x_const.to(device)

    h_star = h_star.detach().clone().to(device).unsqueeze(0)
    h_star.requires_grad_(True)

    def f_of_h(h):
        return one_step_hidden_update(net, h, x_const)

    J = torch.autograd.functional.jacobian(f_of_h, h_star)
    # shape: (1, H, 1, H) -> (H, H)
    J = J[0, :, 0, :]
    return J

In [ ]:
import numpy as np
import torch

# =========================================================
# Helper: spectral radius of Jacobian at each fixed point
# =========================================================
def fixed_point_spectral_radius(net, h_star, x_const):
    J = fixed_point_jacobian(net, h_star, x_const)
    eigvals = torch.linalg.eigvals(J)
    rho = eigvals.abs().max().item()
    return rho, eigvals

def find_fps_from_saved_hidden(
    infos,
    model_names,
    envs,
    env_vecs,
    ob_size,
    act_size,
    hidden_size,
    hidden_state_all,   # shape: (n_model, n_seed, n_env, n_trial, T_phase, H)
    phase_inputs_mean,  # shape: (n_env, input_size), you prepare this separately
    device='cpu',
    n_iters=2000,
    lr=1e-2,
    tol=1e-8,
    n_init_per_env=200,
):
    from net.common import init_model

    fp_results = {}

    for i_model, model_name in enumerate(model_names):
        fp_results[model_name] = {}

        for seed in range(hidden_state_all.shape[1]):
            net = init_model(
                ob_size=ob_size + len(envs),
                hidden_size=hidden_size,
                act_size=act_size,
                model_name=model_name
            ).to(device)

            net.load_state_dict(infos[i_model]['net_state_dicts'][seed][-1][0])
            net.eval()

            for k in range(len(envs)):
                env_name = task_ids[k] if 'task_ids' in globals() else f'env_{k}'
                key = f"{env_name}_seed{seed}"

                h_init_states = hidden_state_all[i_model, seed, k].reshape(-1, hidden_size)

                if len(h_init_states) > n_init_per_env:
                    choice = np.random.choice(len(h_init_states), n_init_per_env, replace=False)
                    h_init_states = h_init_states[choice]

                x_const = torch.tensor(phase_inputs_mean[k], dtype=torch.float32, device=device)

                fps, q_vals, all_h_final, all_q_final = find_fixed_points_from_states(
                    net=net,
                    x_const=x_const,
                    h_init_states=h_init_states,
                    n_iters=n_iters,
                    lr=lr,
                    tol=tol
                )

                rho_list = []
                eigvals_list = []
                for fp_idx in range(len(fps)):
                    rho, eigvals = fixed_point_spectral_radius(net, fps[fp_idx], x_const)
                    rho_list.append(rho)
                    eigvals_list.append(eigvals.detach().cpu().numpy())

                fp_results[model_name][key] = {
                    'fps': fps.detach().cpu(),
                    'q_vals': q_vals.detach().cpu(),
                    'rho': rho_list,
                    'eigvals': eigvals_list,
                    'x_const': x_const.detach().cpu(),
                    'h_init_states': h_init_states,
                    'all_h_final': all_h_final.detach().cpu(),
                    'all_q_final': all_q_final.detach().cpu(),
                }

    return fp_results


def compute_phase_mean_inputs(envs, env_vecs, timing_idx, phase='stimulus', num_trial=20):
    phase_inputs_mean = []

    for k, env in enumerate(envs):
        if phase == 'stimulus':
            idx = timing_idx['stimulus'][k]
        elif phase == 'decision':
            idx = timing_idx['decision'][k]
        else:
            raise ValueError("phase must be 'stimulus' or 'decision'")

        all_inputs_phase = []

        for trial_i in range(num_trial):
            env.seed(trial_i)
            env.new_trial()
            ob = env.ob[idx[0]:idx[1]]  # (T_phase, ob_size)
            ob = np.concatenate(
                [ob, np.tile(env_vecs[k], (ob.shape[0], 1))],
                axis=-1
            )
            all_inputs_phase.append(ob)

        all_inputs_phase = np.concatenate(all_inputs_phase, axis=0)
        phase_inputs_mean.append(all_inputs_phase.mean(axis=0))

    return np.stack(phase_inputs_mean, axis=0)